# AI/ML Cloud Services Cheatsheet

> CLI commands and patterns for AWS SageMaker, Azure Machine Learning, and GCP Vertex AI.

---
## AWS SageMaker

### Setup
```bash
pip install sagemaker boto3
aws configure
```

### Training Jobs (CLI)

```bash
# Create a training job
aws sagemaker create-training-job \
  --training-job-name "my-training-$(date +%Y%m%d)" \
  --algorithm-specification '{
    "TrainingImage": "763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.0-gpu-py310",
    "TrainingInputMode": "File"
  }' \
  --role-arn "arn:aws:iam::123456789:role/SageMakerRole" \
  --resource-config '{
    "InstanceCount": 1,
    "InstanceType": "ml.p3.2xlarge",
    "VolumeSizeInGB": 50
  }' \
  --stopping-condition '{"MaxRuntimeInSeconds": 86400}'

# Check training job status
aws sagemaker describe-training-job --training-job-name "my-training-20240101"

# List training jobs
aws sagemaker list-training-jobs --sort-by CreationTime --sort-order Descending --max-results 10
```

In [ ]:
# SageMaker Python SDK - Training with Spot Instances
import sagemaker
from sagemaker.pytorch import PyTorch

session = sagemaker.Session()
role = sagemaker.get_execution_role()

estimator = PyTorch(
    entry_point="train.py",
    source_dir="src/",
    role=role,
    instance_count=1,
    instance_type="ml.p3.2xlarge",
    framework_version="2.0",
    py_version="py310",
    hyperparameters={
        "epochs": 10,
        "batch-size": 64,
        "learning-rate": 0.001,
    },
    use_spot_instances=True,            # Save up to 90% cost
    max_wait=7200,                       # Max wait for spot
    max_run=3600,                        # Max training time
    checkpoint_s3_uri="s3://bucket/checkpoints/",
)

# Start training
# estimator.fit({"training": "s3://my-bucket/data/train/", "validation": "s3://my-bucket/data/val/"})
print("SageMaker estimator configured with spot instances")

### SageMaker Endpoints

```bash
# Create a model
aws sagemaker create-model --model-name "my-model" \
  --primary-container '{"Image": "...", "ModelDataUrl": "s3://my-bucket/output/model.tar.gz"}' \
  --execution-role-arn "arn:aws:iam::123456789:role/SageMakerRole"

# Create endpoint config
aws sagemaker create-endpoint-config --endpoint-config-name "my-config" \
  --production-variants '[{"VariantName":"primary","ModelName":"my-model","InstanceType":"ml.g4dn.xlarge","InitialInstanceCount":1}]'

# Create endpoint
aws sagemaker create-endpoint --endpoint-name "my-endpoint" --endpoint-config-name "my-config"

# Invoke endpoint
aws sagemaker-runtime invoke-endpoint --endpoint-name "my-endpoint" \
  --content-type "application/json" --body '{"instances": [[1.0, 2.0, 3.0, 4.0]]}' output.json

# Delete endpoint (stop billing)
aws sagemaker delete-endpoint --endpoint-name "my-endpoint"
```

---
## Azure Machine Learning

### Setup

```bash
# Install Azure ML CLI extension
az extension add -n ml

# Create workspace
az ml workspace create --name my-ml-workspace --resource-group my-rg --location eastus

# Set defaults
az configure --defaults group=my-rg workspace=my-ml-workspace
```

### Compute

```bash
# Create compute cluster (for training)
az ml compute create --name gpu-cluster --type AmlCompute \
  --size Standard_NC6s_v3 --min-instances 0 --max-instances 4 \
  --idle-time-before-scale-down 300

# Create compute instance (for development)
az ml compute create --name dev-instance --type ComputeInstance --size Standard_DS3_v2

# List compute resources
az ml compute list -o table

# Stop compute instance (save costs)
az ml compute stop --name dev-instance
```

### Training Jobs

```yaml
# job.yml
$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
command: python train.py --epochs ${{inputs.epochs}} --lr ${{inputs.learning_rate}}
environment:
  image: mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04
  conda_file: conda.yml
compute: azureml:gpu-cluster
inputs:
  epochs: 10
  learning_rate: 0.001
code: ./src
experiment_name: my-experiment
```

```bash
# Submit / monitor / download
az ml job create -f job.yml
az ml job show --name <job-name> -o table
az ml job stream --name <job-name>
az ml job download --name <job-name> --output-name model
az ml job cancel --name <job-name>
```

### Hyperparameter Sweep

```yaml
# sweep.yml
type: sweep
trial:
  command: python train.py --lr ${{search_space.lr}} --batch-size ${{search_space.batch_size}}
  environment:
    image: mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04
  code: ./src
  compute: azureml:gpu-cluster
search_space:
  lr:
    type: loguniform
    min_value: -5
    max_value: -1
  batch_size:
    type: choice
    values: [16, 32, 64, 128]
objective:
  primary_metric: val_accuracy
  goal: maximize
sampling_algorithm: bayesian
limits:
  max_total_trials: 20
  max_concurrent_trials: 4
```

### Online Endpoints

```bash
# Create endpoint
az ml online-endpoint create -f endpoint.yml

# Create deployment
az ml online-deployment create -f deployment.yml --all-traffic

# Test endpoint
az ml online-endpoint invoke --name fraud-endpoint --request-file sample-request.json

# Scale deployment
az ml online-deployment update --name blue --endpoint fraud-endpoint --instance-count 3

# Get credentials
az ml online-endpoint get-credentials --name fraud-endpoint

# Delete endpoint (stop billing)
az ml online-endpoint delete --name fraud-endpoint --yes
```

---
## GCP Vertex AI

### Setup
```bash
pip install google-cloud-aiplatform
gcloud auth application-default login
gcloud config set project my-project-id
```

In [ ]:
# Vertex AI Python SDK - Custom Training (reference code)
from google.cloud import aiplatform

aiplatform.init(project="my-project", location="us-central1")

job = aiplatform.CustomTrainingJob(
    display_name="my-training",
    script_path="train.py",
    container_uri="us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-0:latest",
    requirements=["transformers", "datasets"],
)

# model = job.run(
#     replica_count=1,
#     machine_type="n1-standard-8",
#     accelerator_type="NVIDIA_TESLA_V100",
#     accelerator_count=1,
# )
print("Vertex AI training job configured")

### Vertex AI Endpoints

```bash
# Upload model
gcloud ai models upload --region=us-central1 --display-name="fraud-model" \
  --container-image-uri="us-docker.pkg.dev/vertex-ai/prediction/pytorch-gpu.2-0:latest" \
  --artifact-uri="gs://my-bucket/model/"

# Create endpoint
gcloud ai endpoints create --region=us-central1 --display-name="fraud-endpoint"

# Deploy model to endpoint
gcloud ai endpoints deploy-model ENDPOINT_ID --region=us-central1 --model=MODEL_ID \
  --display-name="fraud-v1" --machine-type=n1-standard-4 \
  --accelerator-type=NVIDIA_TESLA_T4 --accelerator-count=1

# Predict
gcloud ai endpoints predict ENDPOINT_ID --region=us-central1 --json-request=request.json

# Undeploy (stop billing)
gcloud ai endpoints undeploy-model ENDPOINT_ID --region=us-central1 --deployed-model-id=ID
```

---
## Cloud Comparison Matrix

| Feature | AWS SageMaker | Azure ML | GCP Vertex AI |
|---------|--------------|----------|---------------|
| **Managed Notebooks** | SageMaker Studio | Compute Instances | Workbench |
| **Training** | Training Jobs | Command Jobs | Custom Jobs |
| **HPO** | Automatic Model Tuning | Sweep Jobs | Vizier |
| **Model Registry** | Model Registry | Model Registry | Model Registry |
| **Real-time Inference** | Endpoints | Online Endpoints | Endpoints |
| **Batch Inference** | Batch Transform | Batch Endpoints | Batch Prediction |
| **Pipelines** | SageMaker Pipelines | Azure ML Pipelines | Vertex Pipelines |
| **Feature Store** | Feature Store | Managed Feature Store | Feature Store |
| **AutoML** | Autopilot | AutoML | AutoML |
| **GPU Cheapest** | ml.g4dn.xlarge (~$0.53/hr) | Standard_NC4as_T4_v3 (~$0.53/hr) | n1-standard-4 + T4 (~$0.55/hr) |

## Cost Optimization Tips

```bash
# AWS: Use spot instances (up to 90% savings)
# Set use_spot_instances=True in SageMaker estimator

# Azure: Use low-priority VMs
az ml compute create --name gpu-cluster --type AmlCompute \
  --size Standard_NC6s_v3 --tier low_priority

# GCP: Use preemptible VMs
# Add --preemptible flag to compute creation

# All clouds: Auto-scale to zero when idle (min-instances 0)
# All clouds: Right-size instances (T4 for inference, V100/A100 for training)
```

## Interview Scenarios

**Q: How would you choose between AWS SageMaker, Azure ML, and GCP Vertex AI?**
> Consider: (1) existing cloud investment and team expertise, (2) specific features needed (Azure ML has best MLflow integration, SageMaker has broadest instance selection, Vertex AI has best AutoML), (3) pricing for your workload pattern, (4) compliance requirements, (5) integration with other services.

**Q: How do you manage costs for ML workloads in the cloud?**
> Key strategies: spot/preemptible instances for training (with checkpointing), auto-scale to zero for compute clusters, right-size GPU selection, reserved instances for steady-state inference, batch inference instead of real-time where possible, and use cloud cost monitoring tools.